<a href="https://colab.research.google.com/github/lcambisaca/IR-experiments/blob/main/01_visdial_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PlugIR quick start: small VisDial pool

**Default: 5,000 candidate images (10%), 32 matched target queries, 11 query states.** At the measured speed of roughly 1.1 seconds per batch of 8, indexing is approximately 11-12 minutes, excluding setup and backups. Actual time varies.

All test targets are retained. The remaining candidates are fixed random distractors with seed 42. This is useful for development and comparing methods under the same small pool; its easier retrieval scores are not full benchmark results.

No LLM/OpenRouter calls, no training, and no additional benchmark downloads. The existing full archive stays unchanged. A separate small archive and compatible image-vector cache are saved for teammates. If a compatible cache exists, raw image restoration is skipped for these text-query evaluations.

This is a separate notebook so it does not alter a full-pool run in progress. Stop the old evaluation before starting this one; an unfinished full-pool index cannot be reused because upstream indexing saves only at completion. Keep the same runtime if you want to reuse its local image files.

[Original repository](https://github.com/Saehyung-Lee/PlugIR) · [Paper](https://arxiv.org/abs/2406.03411)

## 1. Settings
Use 5,000 for the quick start. Set `N_IMAGES = 12500` for 25% (roughly 29 minutes of encoding at the observed speed). Keep settings identical across teammates when comparing results. Do not change the pool while a run is active.

In [1]:
from pathlib import Path

# Team settings: adjust the Drive folder to match your own shortcut.
REPO_DIR = Path('/content/PlugIR')
DATA_DIR = Path('/content/plugir_data')
SHARED_DIR = Path('/content/drive/MyDrive/PlugIR')
ARCHIVE_ON_DRIVE = SHARED_DIR / 'plugir_corpus_50k.tar'

# Small VisDial development pool. Use 12500 for 25%, or 50000 for the full pool.
N_IMAGES = 5000
POOL_SEED = 42
N_QUERIES = 32
N_ROUNDS = 11                 # Caption plus ten question-answer rounds
TOP_K = 10
IMAGE_BATCH_SIZE = 8
QUERY_BATCH_SIZE = 8
NUM_WORKERS = 2
UPSTREAM_COMMIT = 'ce79c9ff834aabc9de698efb706c6cb9427ecf35'           # After the first run, set this to the printed commit.
RUN_TAG = f'visdial_{N_IMAGES}_first{N_QUERIES}_seed{POOL_SEED}'     # Change this when changing experiment settings.


## 2. Dependencies
For a runtime that already has these versions installed, skip installation. If a restart is requested, restart the session and rerun settings.

In [2]:
# Retain Colab's CUDA-enabled PyTorch and torchvision.
%pip install -q --only-binary=tokenizers "transformers==4.51.3" "tokenizers==0.21.4" "openai==0.28.1" accelerate sentencepiece fast-pytorch-kmeans tqdm pillow ftfy regex requests
%pip install -q git+https://github.com/openai/CLIP.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Preparing metadata (setup.py) ... done


## 3. GPU, code and Drive
If using the current runtime, existing image files remain available. Each teammate needs access to the shared Drive folder or a shortcut at the configured path.

In [4]:
import json
import os
import subprocess
import sys
import torch
import torchvision
import transformers
import clip
from transformers import AutoProcessor, BlipForImageTextRetrieval
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('Transformers:', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0))

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Saehyung-Lee/PlugIR.git', str(REPO_DIR)], check=True)
if UPSTREAM_COMMIT:
    subprocess.run(['git', 'checkout', '--detach', UPSTREAM_COMMIT], cwd=REPO_DIR, check=True)
UPSTREAM_VERSION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('PlugIR commit:', UPSTREAM_VERSION)

drive.mount('/content/drive')
if not SHARED_DIR.is_dir():
    raise FileNotFoundError('Shared Drive folder not found. Check access and SHARED_DIR.')


Python: 3.13.15
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
Transformers: 4.51.3
GPU: Tesla T4
PlugIR commit: ce79c9ff834aabc9de698efb706c6cb9427ecf35
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Choose matched queries and the small pool
The actual evaluator corpus path is changed to the saved subset manifest; reducing only the query count would not reduce image encoding. Original target-to-caption associations and corpus order are preserved. The official evaluator is adapted for Colab, while metric formulas remain unchanged.

In [5]:
#@title Prepare evaluator and matched query files
import hashlib
import importlib.metadata

def patch_evaluator(source, image_batch_size=8, num_workers=2):
    # Keep the original ranking and metric calculations. These edits address
    # device placement, model mode, batch sizes, and unused ranking storage.
    replacements = [
        ("'corpus_bs': 256", f"'corpus_bs': {image_batch_size}"),
        ("'num_workers': 8", f"'num_workers': {num_workers}"),
        ('model = model.to(device)', 'model = model.to(device).eval()'),
        ('return_tensors="pt"))', 'return_tensors="pt").to(device))'),
        ('ranks.append(arg_ranks)', 'ranks.append(torch.empty(0, dtype=torch.long))'),
        ('self.scores[i] = self.scores[i].cpu()', 'del self.scores[i]'),
        ("torch.save(self.corpus, self.cfg['cache_corpus'])",
         "torch.save(self.corpus, self.cfg['cache_corpus'] + '.writing'); os.replace(self.cfg['cache_corpus'] + '.writing', self.cfg['cache_corpus'])"),
    ]
    for old, new in replacements:
        if old not in source:
            raise RuntimeError(f'Upstream code changed; review before patching: {old}')
        source = source.replace(old, new)
    compile(source, 'eval_colab.py', 'exec')
    return source

def prepare_query_pair(baseline, plugir, n_queries, n_rounds, corpus_paths):
    if not 1 <= n_queries <= 64:
        raise ValueError('This starter supports 1-64 queries; review memory and scope before scaling.')
    if not 2 <= n_rounds <= 11:
        raise ValueError('Use 2-11 rounds; BRI needs at least two points.')
    by_image = {}
    for q in plugir:
        image = q['img']
        if image in by_image:
            raise ValueError(f'Duplicate PlugIR target: {image}')
        by_image[image] = q
    pairs = []
    seen = set()
    for q in baseline:
        target = q['img']
        if target in seen or target not in by_image:
            continue
        other = by_image[target]
        if target not in corpus_paths:
            raise ValueError(f'Target missing from corpus: {target}')
        if len(q['dialog']) < n_rounds or len(other['dialog']) < n_rounds:
            continue
        if q['dialog'][0].strip().lower() != other['dialog'][0].strip().lower():
            raise ValueError(f'Initial captions differ for target {target}; review pairing.')
        pairs.append((q, other))
        seen.add(target)
        if len(pairs) == n_queries:
            break
    if len(pairs) != n_queries:
        raise ValueError(f'Only {len(pairs)} matched examples have enough rounds.')
    return [p[0] for p in pairs], [p[1] for p in pairs]


def select_image_pool(full_paths, targets, n_images, seed):
    import random
    if len(full_paths) != len(set(full_paths)):
        raise ValueError('Duplicate corpus paths.')
    targets = set(targets)
    if not targets.issubset(set(full_paths)):
        raise ValueError('A target is outside the original corpus.')
    if not len(targets) <= n_images <= len(full_paths):
        raise ValueError('Pool must fit all targets and not exceed the full corpus.')
    distractors = [p for p in full_paths if p not in targets]
    chosen = targets | set(random.Random(seed).sample(distractors, n_images-len(targets)))
    # Preserve original ordering so embeddings and paths stay aligned.
    return [p for p in full_paths if p in chosen]

raw_source = (REPO_DIR / 'eval.py').read_text()
patched_source = patch_evaluator(raw_source, IMAGE_BATCH_SIZE, NUM_WORKERS)
(REPO_DIR / 'eval_colab.py').write_text(patched_source)

manifest_path = REPO_DIR / 'Protocol/Search_Space_val_50k.json'
corpus_paths = json.loads(manifest_path.read_text())
if len(corpus_paths) != 50000:
    raise ValueError('Expected the official 50,000-image corpus.')
baseline = json.loads((REPO_DIR / 'dialogues/VisDial_v1.0_queries_val.json').read_text())
plugir_files = list((REPO_DIR / 'dialogues').glob('ours_final_*_reconed.json'))
if len(plugir_files) != 1:
    raise RuntimeError('Expected one supplied reformulated query file; select the intended file explicitly.')
plugir = json.loads(plugir_files[0].read_text())
baseline_subset, plugir_subset = prepare_query_pair(baseline, plugir, N_QUERIES, N_ROUNDS, set(corpus_paths))


full_corpus_paths = corpus_paths
corpus_paths = select_image_pool(
    full_corpus_paths, [q['img'] for q in baseline_subset], N_IMAGES, POOL_SEED
)
manifest_path = REPO_DIR / 'Protocol/team_search_space.json'
manifest_path.write_text(json.dumps(corpus_paths))
old_corpus_setting = "'corpus_path': 'Protocol/Search_Space_val_50k.json'"
if old_corpus_setting not in patched_source:
    raise RuntimeError('Corpus setting changed; review the evaluator adapter.')
patched_source = patched_source.replace(old_corpus_setting, "'corpus_path': 'Protocol/team_search_space.json'")
compile(patched_source, 'eval_colab.py', 'exec')

BASELINE_PATH = REPO_DIR / 'dialogues/team_baseline.json'
PLUGIR_PATH = REPO_DIR / 'dialogues/team_plugir.json'
BASELINE_PATH.write_text(json.dumps(baseline_subset))
PLUGIR_PATH.write_text(json.dumps(plugir_subset))

# Include model revision and preprocessing/software versions in cache identity.
from transformers import AutoConfig
MODEL_ID = 'Salesforce/blip-itm-large-coco'
model_config = AutoConfig.from_pretrained(MODEL_ID)
MODEL_REVISION = getattr(model_config, '_commit_hash', None)
if not MODEL_REVISION:
    raise RuntimeError('Could not resolve a model revision for cache tracking.')
old_call = 'from_pretrained("' + MODEL_ID + '")'
new_call = 'from_pretrained("' + MODEL_ID + '", revision=' + repr(MODEL_REVISION) + ')'
if old_call not in patched_source:
    raise RuntimeError('Model load call changed; inspect the evaluator.')
patched_source = patched_source.replace(old_call, new_call)
compile(patched_source, 'eval_colab.py', 'exec')
(REPO_DIR / 'eval_colab.py').write_text(patched_source)

cache_identity = {
    'model': MODEL_ID, 'model_revision': MODEL_REVISION,
    'manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    'transformers': transformers.__version__, 'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'pillow': importlib.metadata.version('pillow'),
    'adapter_sha256': hashlib.sha256(patched_source.encode()).hexdigest(),
    'mode': 'eval',
}
fingerprint = hashlib.sha256(json.dumps(cache_identity, sort_keys=True).encode()).hexdigest()[:16]
CACHE_PATH = REPO_DIR / 'cache' / f'corpus_blip_eval_{fingerprint}.pth'
CACHE_PATH.parent.mkdir(exist_ok=True)
SHARED_CACHE_DIR = SHARED_DIR / 'cache'
SHARED_CACHE_DIR.mkdir(exist_ok=True)

RUN_DIR = REPO_DIR / 'logs' / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_INFO = {
    'upstream_commit': UPSTREAM_VERSION, 'cache_identity': cache_identity,
    'queries': N_QUERIES, 'rounds': N_ROUNDS, 'K': TOP_K,
    'candidate_images': N_IMAGES, 'pool_seed': POOL_SEED,
    'image_batch_size': IMAGE_BATCH_SIZE, 'query_batch_size': QUERY_BATCH_SIZE,
    'baseline_sha256': hashlib.sha256(BASELINE_PATH.read_bytes()).hexdigest(),
    'plugir_sha256': hashlib.sha256(PLUGIR_PATH.read_bytes()).hexdigest(),
    'targets': [q['img'] for q in baseline_subset],
    'scope': 'Reduced VisDial image pool; development only; supplied query evaluation; no API calls',
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(RUN_INFO, indent=2))
print(f'Matched {N_QUERIES} targets; {N_ROUNDS} rounds; {len(corpus_paths):,} candidate images.')
print('Reduced image pool: results are not comparable to the full paper benchmark.')
print('Corpus cache:', CACHE_PATH.name)
print('The supplied PlugIR queries were generated previously by the authors.')
print('The VisDial-dialogue baseline is not a new generation of the paper’s ZS questioner.')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Matched 32 targets; 11 rounds; 5,000 candidate images.
Reduced image pool: results are not comparable to the full paper benchmark.
Corpus cache: corpus_blip_eval_104ac21eef4631e7.pth
The supplied PlugIR queries were generated previously by the authors.
The VisDial-dialogue baseline is not a new generation of the paper’s ZS questioner.


## 5. Reuse or create a separate small-pool cache
A full-pool cache is not overwritten. Cache identity includes the selected manifest, model revision and software settings. Live output shows stages and progress.

In [6]:
#@title Cache and evaluation helpers
import queue
import re
import subprocess
import threading
import time


def run_with_live_output(command, cwd, env, output_log):
    """Stream a child process to Colab; show a heartbeat during silent stages."""
    print('Starting evaluator. Model loading comes before image indexing.', flush=True)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    messages = queue.Queue()

    def read_output():
        try:
            for line in process.stdout:
                messages.put(line)
        finally:
            messages.put(None)

    reader = threading.Thread(target=read_output, daemon=True)
    reader.start()
    started = time.monotonic()
    last_display = started
    last_progress = started - 10
    finished = False
    try:
        with open(output_log, 'w', encoding='utf-8') as stream:
            while not finished:
                try:
                    line = messages.get(timeout=1)
                except queue.Empty:
                    now = time.monotonic()
                    if now - last_display >= 20:
                        print(f'Evaluator active ({int(now-started)}s). No new output; stage progress unknown.', flush=True)
                        last_display = now
                    continue
                if line is None:
                    finished = True
                    continue
                stream.write(line)
                stream.flush()
                now = time.monotonic()
                is_progress = bool(re.search(r'\d+%\|', line))
                if not is_progress or now - last_progress >= 5:
                    print(line.rstrip(), flush=True)
                    last_display = now
                    if is_progress:
                        last_progress = now
        result = process.wait()
        if result:
            raise subprocess.CalledProcessError(result, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        raise
    finally:
        process.stdout.close()


def enable_evaluator_diagnostics(script_path):
    """Add logging only; keep embeddings/cache identity unchanged."""
    source = script_path.read_text()
    source = source.replace('logging.basicConfig(', 'logging.basicConfig(force=True,', 1) if 'logging.basicConfig(force=True,' not in source else source
    if 'Loading BLIP model weights...' not in source:
        source = source.replace(
            '    model = BlipForRetrieval.from_pretrained(',
            "    print('Loading BLIP model weights...', flush=True)\n    model = BlipForRetrieval.from_pretrained(", 1)
    if 'Starting image indexing or cache load...' not in source:
        source = source.replace(
            '    evaluator.index_corpus()',
            "    print('Starting image indexing or cache load...', flush=True)\n    evaluator.index_corpus()", 1)
    compile(source, str(script_path), 'exec')
    script_path.write_text(source)


import gc
import shutil
import subprocess
import sys

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_complete(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.uploading')
    shutil.copyfile(source, temporary)
    if file_sha256(temporary) != file_sha256(source):
        raise RuntimeError(f'Copy verification failed: {destination.name}')
    temporary.replace(destination)

def check_cache(path):
    ids, vectors = torch.load(path, map_location='cpu', weights_only=True)
    if ids.shape != (len(corpus_paths),) or vectors.shape != (len(corpus_paths), 256):
        raise ValueError('Unexpected BLIP corpus cache shape.')
    if not torch.equal(ids, torch.arange(len(corpus_paths))) or not torch.isfinite(vectors).all():
        raise ValueError('Invalid corpus IDs or non-finite vectors.')

def restore_cache_if_available():
    shared = SHARED_CACHE_DIR / CACHE_PATH.name
    meta = shared.with_suffix('.json')
    if CACHE_PATH.is_file():
        check_cache(CACHE_PATH)
        print('Using local cache:', CACHE_PATH.name)
        return
    if shared.is_file() and meta.is_file():
        saved = json.loads(meta.read_text())
        if saved['identity'] != cache_identity:
            raise ValueError('Shared cache metadata do not match this model/setup.')
        copy_complete(shared, CACHE_PATH)
        if file_sha256(CACHE_PATH) != saved['sha256']:
            raise ValueError('Shared cache checksum does not match metadata.')
        check_cache(CACHE_PATH)
        print('Restored verified cache from Drive.')
    else:
        print(f'No compatible shared cache yet. First run will encode {len(corpus_paths):,} selected images.')

def save_cache():
    if not CACHE_PATH.is_file():
        return
    check_cache(CACHE_PATH)
    shared = SHARED_CACHE_DIR / CACHE_PATH.name
    metadata = {'identity': cache_identity, 'sha256': file_sha256(CACHE_PATH)}
    shared_meta = shared.with_suffix('.json')
    if shared.is_file() and shared_meta.is_file() and json.loads(shared_meta.read_text()) == metadata:
        return
    copy_complete(CACHE_PATH, shared)
    local_meta = CACHE_PATH.with_suffix('.json')
    local_meta.write_text(json.dumps(metadata, indent=2))
    copy_complete(local_meta, shared_meta)
    print('Image embeddings backed up to Drive.')

def run_evaluation(label, query_path, split=False):
    gc.collect()
    torch.cuda.empty_cache()
    destination = RUN_DIR / label
    destination.mkdir(exist_ok=True)
    log = destination / 'test.log'
    completed = destination / 'complete.json'
    enable_evaluator_diagnostics(REPO_DIR / 'eval_colab.py')
    RUN_INFO['diagnostic_adapter_sha256'] = file_sha256(REPO_DIR / 'eval_colab.py')
    (RUN_DIR / 'run_config.json').write_text(json.dumps(RUN_INFO, indent=2))
    signature = {'run': RUN_INFO, 'label': label, 'split': split}
    if completed.is_file() and json.loads(completed.read_text()) == signature and log.is_file():
        print(f'{label}: completed result already exists. Change RUN_TAG to run again.')
        return
    # The upstream logger appends; clear only this incomplete run's log.
    if log.exists():
        log.write_text('')
    command = [sys.executable, '-u', str(REPO_DIR / 'eval_colab.py'),
        '--retriever', 'blip', '--data-dir', str(DATA_DIR),
        '--queries-path', str(query_path), '--cache-corpus', str(CACHE_PATH),
        '--batch-size', str(QUERY_BATCH_SIZE), '--num-rounds', str(N_ROUNDS),
        '--K', str(TOP_K), '--output-dir', str(destination)]
    if split:
        command.append('--split')
    environment = os.environ.copy()
    environment['USE_TF'] = '0'  # This workflow uses PyTorch only.
    environment['TOKENIZERS_PARALLELISM'] = 'false'
    try:
        run_with_live_output(command, cwd=REPO_DIR, env=environment, output_log=destination / 'console.log')
        completed.write_text(json.dumps(signature, indent=2))
    finally:
        # A finished index can be preserved even if later evaluation fails.
        save_cache()
        for file in RUN_DIR.rglob('*'):
            if file.is_file():
                copy_complete(file, SHARED_DIR / 'results' / RUN_TAG / file.relative_to(RUN_DIR))
    print('Saved logs and configuration to Drive.')

restore_cache_if_available()


No compatible shared cache yet. First run will encode 5,000 selected images.


## 6. Prepare only the images needed
If a matching cache was restored, evaluation needs no image pixels. Otherwise reuse local files; on a fresh runtime, prefer the saved small archive, falling back to the existing full backup once. The first preparer creates the smaller archive for teammates. These file checks do not decode every JPEG.

In [7]:
#@title Subset image and archive helper
import json
from pathlib import Path
import shutil
import tarfile
from tqdm.auto import tqdm


def prepare_subset_images(expected_paths, data_dir, shared_dir, pool_hash):
    """Reuse local images, or restore only this pool; save a compact team archive."""
    data_dir, shared_dir = Path(data_dir), Path(shared_dir)
    expected_paths = list(expected_paths)
    data_dir.mkdir(parents=True, exist_ok=True)
    root = data_dir.resolve()
    for name in expected_paths:
        if Path(name).is_absolute() or not (data_dir / name).resolve().is_relative_to(root):
            raise ValueError(f'Unsafe image path: {name}')
    pool_dir = shared_dir / 'subsets'
    pool_dir.mkdir(parents=True, exist_ok=True)
    name = f'visdial_{len(expected_paths)}_{pool_hash[:16]}'
    shared_archive = pool_dir / (name + '.tar')
    shared_manifest = pool_dir / (name + '.json')
    needs_restore = any(not (data_dir / p).is_file() or (data_dir / p).stat().st_size == 0 for p in expected_paths)

    if needs_restore:
        # The small archive is preferred. Fall back to the full backup once.
        source = shared_archive if shared_archive.is_file() else shared_dir / 'plugir_corpus_50k.tar'
        if not source.is_file():
            raise FileNotFoundError(f'Archive not found: {source}')
        local = data_dir.parent / source.name
        if not local.is_file() or local.stat().st_size != source.stat().st_size:
            if shutil.disk_usage(data_dir).free < source.stat().st_size + 1024**3:
                raise RuntimeError('Insufficient space to copy the source archive.')
            print(f'Copying {source.name} from Drive ({source.stat().st_size/1e9:.2f} GB)...', flush=True)
            temporary = local.with_suffix('.copying')
            shutil.copyfile(source, temporary)
            if temporary.stat().st_size != source.stat().st_size:
                raise RuntimeError('Archive copy was incomplete.')
            temporary.replace(local)
        with tarfile.open(local, 'r') as archive:
            members = archive.getmembers()
            wanted = set(expected_paths)
            selected = [m for m in members if m.name in wanted]
            if len(selected) != len(wanted) or {m.name for m in selected} != wanted:
                raise ValueError('Archive does not contain each selected image exactly once.')
            if any(not m.isfile() for m in selected):
                raise ValueError('Expected regular image files, not links or folders.')
            required = sum(m.size for m in selected if not (data_dir / m.name).is_file())
            if shutil.disk_usage(data_dir).free < required + 1024**3:
                raise RuntimeError('Insufficient space to restore selected images.')
            for member in tqdm(selected, desc='Restore selected images'):
                destination = data_dir / member.name
                if destination.is_file() and destination.stat().st_size == member.size:
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_suffix('.restoring')
                with archive.extractfile(member) as src, temporary.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
                if temporary.stat().st_size != member.size:
                    raise RuntimeError(f'Incomplete image: {member.name}')
                temporary.replace(destination)
    else:
        print('Selected images are already on local disk; no copy or extraction needed.')

    if any(not (data_dir / p).is_file() or (data_dir / p).stat().st_size == 0 for p in expected_paths):
        raise RuntimeError('Some selected images are missing or empty.')

    if shared_archive.is_file() and shared_manifest.is_file() and json.loads(shared_manifest.read_text()) == expected_paths:
        print('A matching small archive is already available for teammates.')
        return

    local_small = data_dir.parent / (name + '.tar')
    needed = sum((data_dir / p).stat().st_size + 1024 for p in expected_paths) + 1024**3
    if shutil.disk_usage(data_dir).free < needed:
        raise RuntimeError('Insufficient disk space to create the small team archive.')
    with tarfile.open(local_small, 'w') as archive:
        for path in tqdm(expected_paths, desc='Package small pool for team'):
            archive.add(data_dir / path, arcname=path)
    print('Saving the small archive to Drive...', flush=True)
    temporary = shared_archive.with_suffix('.uploading')
    shutil.copyfile(local_small, temporary)
    if temporary.stat().st_size != local_small.stat().st_size:
        raise RuntimeError('Small archive copy was incomplete.')
    temporary.replace(shared_archive)
    manifest_temporary = shared_manifest.with_suffix('.writing')
    manifest_temporary.write_text(json.dumps(expected_paths))
    manifest_temporary.replace(shared_manifest)
    print(f'Team subset saved: {shared_archive.name} ({shared_archive.stat().st_size/1e9:.2f} GB)')


In [8]:
if CACHE_PATH.is_file():
    print('Verified cache is ready; skipping raw-image restoration for text evaluation.')
else:
    prepare_subset_images(corpus_paths, DATA_DIR, SHARED_DIR, cache_identity['manifest_sha256'])


Copying plugir_corpus_50k.tar from Drive (8.22 GB)...


Restore selected images:   0%|          | 0/5000 [00:00<?, ?it/s]

Package small pool for team:   0%|          | 0/5000 [00:00<?, ?it/s]

Saving the small archive to Drive...
Team subset saved: visdial_5000_c4087b6d5fed216b.tar (0.83 GB)


## 7. Run baseline
First encoding pass: 625 batches at the default settings, not 6,250. Uses provided VisDial dialogue; this is not regeneration of the paper’s ZS questioner.

In [9]:
run_evaluation('baseline', BASELINE_PATH)


Starting evaluator. Model loading comes before image indexing.
Loading BLIP model weights...
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Starting image indexing or cache load...
/content/PlugIR/cache/corpus_blip_eval_104ac21eef4631e7.pth
False
2026-10-19 22:41:34,268 | Preparing corpus (search space)...

100%|██████████| 625/625 [12:47<00:00,  1.25s/it]
2026-10-19 22:54:21,850 | Calculate recalls for each dialogues of length 0...

2026-10-19 22:54:22,149 | Calculate recalls for each dialogues of length 1...

2026-10-19 22:54:22,392 | Calculate recalls for each dialogues of length 2...

2026-10-19 22:54:22,630 | Calculate recalls for each dialogues of length 3...

2026-10-19 22:54:22,886 | Calculate recalls for 

## 8. Run supplied PlugIR queries
Same small image pool and target examples. Reuses the baseline’s image vectors. No LLM credits.

In [10]:
run_evaluation('plugir', PLUGIR_PATH, split=True)


Starting evaluator. Model loading comes before image indexing.
Loading BLIP model weights...
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Starting image indexing or cache load...
/content/PlugIR/cache/corpus_blip_eval_104ac21eef4631e7.pth
True
2026-10-19 22:54:54,390 | <<<<Cached corpus has been loaded: /content/PlugIR/cache/corpus_blip_eval_104ac21eef4631e7.pth >>>>>
2026-10-19 22:54:54,390 | Warning: Make sure this corpus has been indexed with the right image embedder!
2026-10-19 22:54:54,404 | Calculate recalls for each dialogues of length 0...

  0%|          | 0/4 [00:00<?, ?it/s]
2026-10-19 22:54:54,992 | Calculate recalls for each dialogues of length 1...

2026-10-19 22:54:55,179 | Calculate recalls for e

## 9. Compare and save
Recall@10 measures success now; Hits@10 measures success at least once so far; BRI rewards earlier improvement in best rank (lower is better). Both methods use the same pool. Small-pool results do not establish full-benchmark performance.

In [11]:
import re
import pandas as pd
from IPython.display import display

def read_metrics(log_path, k, rounds):
    text = Path(log_path).read_text()
    def read_section(name):
        marker = f'====== Results for {name}@{k} ======'
        if marker not in text:
            raise RuntimeError(f'Missing {name} results: evaluation may not have completed.')
        section = text.split(marker, 1)[1].split('======', 1)[0]
        values = {int(r): float(v) for r, v in re.findall(r'Dialog Length: (\d+): ([0-9.]+)%', section)}
        if set(values) != set(range(rounds)):
            raise RuntimeError(f'Expected {rounds} rounds in {name} results.')
        return [values[i] for i in range(rounds)]
    hits = read_section('Hits')
    recall = read_section('Recall')
    matches = re.findall(r'BRI:\s*([0-9.eE+-]+)', text)
    if not matches:
        raise RuntimeError('BRI result missing.')
    return {'hits': hits, 'recall': recall, 'bri': float(matches[-1])}

summary, rows = [], []
for label, title in [('baseline', 'VisDial raw dialogue'), ('plugir', 'Supplied PlugIR queries')]:
    result = read_metrics(RUN_DIR / label / 'test.log', TOP_K, N_ROUNDS)
    summary.append({'Method': title, f'Final Recall@{TOP_K} (%)': result['recall'][-1],
                    f'Final Hits@{TOP_K} (%)': result['hits'][-1], 'BRI (lower is better)': result['bri']})
    for round_id in range(N_ROUNDS):
        rows.append({'Method': title, 'Round': round_id,
                     f'Recall@{TOP_K} (%)': result['recall'][round_id],
                     f'Hits@{TOP_K} (%)': result['hits'][round_id]})
summary_table = pd.DataFrame(summary)
round_table = pd.DataFrame(rows)
display(summary_table)
display(round_table)
for name, table in [('summary.csv', summary_table), ('per_round.csv', round_table)]:
    file = RUN_DIR / name
    table.to_csv(file, index=False)
    copy_complete(file, SHARED_DIR / 'results' / RUN_TAG / name)
print('Reduced-pool development results only; do not compare these scores with the full 50k benchmark.')


,Method,Final Recall@10 (%),Final Hits@10 (%),BRI (lower is better)
0,VisDial raw dialogue,84.38,100.0,0.391020
1,Supplied PlugIR queries,93.75,100.0,0.249757


,Method,Round,Recall@10 (%),Hits@10 (%)
0,VisDial raw dialogue,0,90.62,90.62
1,VisDial raw dialogue,1,90.62,93.75
2,VisDial raw dialogue,2,93.75,96.88
3,VisDial raw dialogue,3,93.75,96.88
4,VisDial raw dialogue,4,100.00,100.00
5,VisDial raw dialogue,5,93.75,100.00
6,VisDial raw dialogue,6,90.62,100.00
7,VisDial raw dialogue,7,96.88,100.00
8,VisDial raw dialogue,8,87.50,100.00
9,VisDial raw dialogue,9,90.62,100.00


Reduced-pool development results only; do not compare these scores with the full 50k benchmark.


## Team reuse
After seeing **Image embeddings backed up to Drive**, teammates can open this notebook with matching settings and use the shared cache. Small archives and their image lists are under `PlugIR/subsets`; model caches are under `PlugIR/cache`; metrics and configuration are under `PlugIR/results`. GitHub holds notebooks/code, not image archives or keys. Live question generation and the multi-agent extension remain later, separately budgeted steps.